# M18 — FLY-CL 20k Adaptive locked-test closure (Colab)

Set **Runtime type = GPU** and run from top to bottom. A GPU with at least 15 GB memory is required; A100 or L4 is preferred. The notebook uses `/content` only and does not mount Google Drive. It compares Exact FP32, P2B INT8/FP32, and Adaptive INT8/FP16 on the same FLY projection and WTA codes for all six paired replicates. Test accuracy never selects a method or seed.

For a fresh run, upload exactly `srq_fly_selfcontained_three_dataset_results.zip` when asked. For cross-account resume, set `IMPORT_HANDOFF=True` and upload the latest `m18_handoff_002_replicates.zip` or `m18_handoff_004_replicates.zip`.

In [ ]:
REPO_GIT_URL='https://github.com/ZaPhat206/SOHO-CL.git'
REPO_COMMIT='4cb9b27dec86a2f533cd2caffd23c503f4da1a3c'
WORK_DIR='/content/SOHO-CL'
RUN_ROOT='/content/srq_m18'
SOURCE_DIR=RUN_ROOT+'/sources'
FEATURE_CACHE_DIR=RUN_ROOT+'/feature_cache'
OUTPUT_DIR=RUN_ROOT+'/output'
AUTHORIZATION=RUN_ROOT+'/authorization.json'
CODE_CACHE_ROOT='/content/srq_m18_wta'
CONFIG='configs/srq_generalization_m18_fly20k_adaptive_locked_test.json'
RUNNER='tools/srq_generalization_m18.py'
SELECTION_NAME='srq_fly_selfcontained_three_dataset_results.zip'
SELECTION_SHA='e4b630781ff6f69deaecb63dda9926d256cd6b654ef4b51a682bf3ef94e6490b'
CHECKPOINT='/content/model.safetensors'
CHECKPOINT_SIZE=346284714
CHECKPOINT_SHA='32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'
DATA_ROOT='/content/cifar_source'
FINAL_EXPORT='/content/srq_generalization_m18_fly20k_adaptive_locked_test.zip'
IMPORT_HANDOFF=False  # True only when resuming from a downloaded M18 handoff.
AUTO_DOWNLOAD_HANDOFFS=True  # Downloads checkpoints after replicate 2 and 4.
BATCH_SIZE=128
NUM_WORKERS=2

In [ ]:
# Pinned clean checkout and source verification.
import hashlib,json,os,shutil,subprocess,sys,zipfile
from pathlib import Path
os.environ['PYTHONDONTWRITEBYTECODE']='1'
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF','expandable_segments:True')
def sha_raw(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda:handle.read(1<<20),b''): h.update(block)
    return h.hexdigest()
def sha_source(path): return hashlib.sha256(Path(path).read_bytes().replace(b'\r\n',b'\n')).hexdigest()
def run_visible(command):
    process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,encoding='utf-8',errors='replace',bufsize=1,env={**os.environ,'PYTHONUNBUFFERED':'1','PYTHONDONTWRITEBYTECODE':'1'})
    assert process.stdout is not None
    for line in process.stdout: print(line,end='')
    returncode=process.wait()
    if returncode: raise RuntimeError(f'Command failed ({returncode}): {command}')
os.chdir('/content')
if Path(WORK_DIR).exists(): shutil.rmtree(WORK_DIR)
subprocess.run(['git','clone','--no-checkout','--quiet',REPO_GIT_URL,WORK_DIR],check=True)
subprocess.run(['git','checkout','--detach','--quiet',REPO_COMMIT],cwd=WORK_DIR,check=True)
assert subprocess.check_output(['git','rev-parse','HEAD'],cwd=WORK_DIR,text=True).strip()==REPO_COMMIT
os.chdir(WORK_DIR)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt','huggingface_hub'],check=True)
import torch
assert torch.cuda.is_available(),'Enable a Colab GPU and restart from cell 1.'
gpu_bytes=torch.cuda.get_device_properties(0).total_memory
assert gpu_bytes >= 14_000_000_000,f'M18 width-20k requires >=14 GB GPU memory; found {gpu_bytes/2**30:.2f} GiB.'
EXPECTED_SOURCE={
 'configs/srq_generalization_m18_fly20k_adaptive_locked_test.json':'c5922e5cd2737ab396fab49e2b76b34f0af05d0c429a222b3e574accb92513f6',
 'tools/srq_generalization_m18.py':'0fdb88935f3df36f7df4c3a03d7e5ae7f6f4df60e03dd7a140df0c15623acd62',
 'tests/test_srq_generalization_m18.py':'9bc71699dcd623e5bc36b30e5bf7934e1095807f142373e7bafc31a65a173e02',
 'docs/research/SRQ_GENERALIZATION_M18_PROTOCOL.md':'90a0c789cb09973d7dc1fb9e72544c875e42137ed1b32458923b9ad693fd16c0'}
for relative,expected in EXPECTED_SOURCE.items(): assert sha_source(relative)==expected,(relative,sha_source(relative),expected)
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip()
Path(RUN_ROOT).mkdir(parents=True,exist_ok=True)
print('M18 PINNED SOURCE: PASS | GPU:',torch.cuda.get_device_name(0),f'{gpu_bytes/2**30:.2f} GiB')

In [ ]:
# Upload either the locked selection artifact (fresh run) or one M18 handoff (resume).
from google.colab import files
source_dir=Path(SOURCE_DIR); source_dir.mkdir(parents=True,exist_ok=True)
selection_path=source_dir/SELECTION_NAME
if IMPORT_HANDOFF:
    upload_cwd=Path.cwd(); os.chdir('/content')
    try: uploaded=files.upload()
    finally: os.chdir(upload_cwd)
    candidates=[name for name in uploaded if name.startswith('m18_handoff_') and name.endswith('_replicates.zip')]
    assert len(candidates)==1,f'Upload exactly one M18 handoff; got {list(uploaded)}'
    handoff_path=Path('/content')/candidates[0]; handoff_path.write_bytes(uploaded[candidates[0]])
    with zipfile.ZipFile(handoff_path) as archive:
        names=set(archive.namelist()); assert 'HANDOFF_MANIFEST.json' in names
        manifest=json.loads(archive.read('HANDOFF_MANIFEST.json'))
        assert manifest['repo_commit']==REPO_COMMIT and manifest['selection_sha256']==SELECTION_SHA
        for relative,digest in manifest['files'].items():
            candidate=Path(relative)
            assert not candidate.is_absolute() and '..' not in candidate.parts and relative in names
            payload=archive.read(relative); assert hashlib.sha256(payload).hexdigest()==digest,(relative,digest)
            destination=Path(RUN_ROOT)/candidate; destination.parent.mkdir(parents=True,exist_ok=True); destination.write_bytes(payload)
    print('M18 HANDOFF IMPORTED:',manifest['completed_replicates'],'replicates')
else:
    upload_cwd=Path.cwd(); os.chdir(source_dir)
    try: uploaded=files.upload()
    finally: os.chdir(upload_cwd)
    assert list(uploaded)==[SELECTION_NAME],f'Upload exactly {SELECTION_NAME}; got {list(uploaded)}'
    selection_path.write_bytes(uploaded[SELECTION_NAME])
assert selection_path.is_file() and sha_raw(selection_path)==SELECTION_SHA
assert not subprocess.check_output(['git','status','--porcelain'],cwd=WORK_DIR,text=True).strip()
print('M18 LOCKED TRAIN-ONLY SELECTION ARTIFACT: PASS')

In [ ]:
# Focused correctness tests; no feature or prediction from CIFAR test is read here.
run_visible([sys.executable,'-B','-m','pytest','-vv','--tb=long','-p','no:cacheprovider','tests/test_srq_generalization_m18.py','tests/test_adaptive_analytic_ridge.py'])
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip()
print('M18 PREFLIGHT TESTS: PASS')

In [ ]:
# Download the exact frozen ViT checkpoint only when feature extraction is needed.
cache_dir=Path(FEATURE_CACHE_DIR); cache_dir.mkdir(parents=True,exist_ok=True)
if not (cache_dir/'train.pt').is_file():
    from huggingface_hub import hf_hub_download
    downloaded=hf_hub_download(repo_id='timm/vit_base_patch16_224.augreg2_in21k_ft_in1k',filename='model.safetensors')
    if Path(downloaded).resolve()!=Path(CHECKPOINT).resolve(): shutil.copyfile(downloaded,CHECKPOINT)
    assert Path(CHECKPOINT).stat().st_size==CHECKPOINT_SIZE and sha_raw(CHECKPOINT)==CHECKPOINT_SHA
    Path(DATA_ROOT).mkdir(parents=True,exist_ok=True)
    print('CHECKPOINT READY; torchvision will download CIFAR-100 if needed')
else:
    print('HANDOFF FEATURE CACHE FOUND; checkpoint/data download skipped')

In [ ]:
# Create only frozen TRAIN features on a fresh run. test.pt remains absent.
if not (cache_dir/'train.pt').is_file():
    command=[sys.executable,'-u','tools/experiment_runner.py','--extract-features-only','--extract-train-only','--root',DATA_ROOT,'--backbone-checkpoint',CHECKPOINT,'--backbone-checkpoint-size',str(CHECKPOINT_SIZE),'--backbone-checkpoint-sha256',CHECKPOINT_SHA,'--feature-cache-dir',FEATURE_CACHE_DIR,'--output-dir','/content/unused_m18','--dataset','CIFAR-100','--model-name','vit_base_patch16_224','--data-augmentation','vit','--seed','2025','--num-classes','100','--num-tasks','10','--device','cuda','--batch-size',str(BATCH_SIZE),'--num-workers',str(NUM_WORKERS)]
    run_visible(command)
assert (cache_dir/'train.pt').is_file()
if not IMPORT_HANDOFF: assert not (cache_dir/'test.pt').exists()
print('M18 TRAIN FEATURE CACHE: PASS')

In [ ]:
# Lock source, train cache, six seeds, inherited Ridge, and Adaptive policy before test materialization.
if not Path(AUTHORIZATION).is_file():
    assert not (cache_dir/'test.pt').exists()
    run_visible([sys.executable,'-B',RUNNER,'authorize','--config',CONFIG,'--selection-artifact',str(selection_path),'--feature-cache-dir',FEATURE_CACHE_DIR,'--authorization',AUTHORIZATION])
authorization=json.loads(Path(AUTHORIZATION).read_text())
assert authorization['authorized'] is True and authorization['uses_test_set'] is False
if not IMPORT_HANDOFF: assert not (cache_dir/'test.pt').exists()
print('M18 AUTHORIZATION: PASS',authorization['authorization_id'])

## Authorized test boundary

All choices are now hashed. The next cell may materialize CIFAR-100 test features. Do not edit methods, seeds, Ridge, budget, or retry rules after observing any result.

In [ ]:
# Materialize test features only after authorization (or validate an imported authorized cache).
if not (cache_dir/'test.pt').is_file():
    assert Path(CHECKPOINT).is_file()
run_visible([sys.executable,'-B',RUNNER,'extract-test','--config',CONFIG,'--selection-artifact',str(selection_path),'--feature-cache-dir',FEATURE_CACHE_DIR,'--authorization',AUTHORIZATION,'--root',DATA_ROOT,'--backbone-checkpoint',CHECKPOINT if Path(CHECKPOINT).is_file() else '/content/not-needed-on-restore','--device','cuda','--batch-size',str(BATCH_SIZE),'--num-workers',str(NUM_WORKERS)])
assert (cache_dir/'test.pt').is_file()
print('M18 AUTHORIZED TEST FEATURE CACHE: PASS')

In [ ]:
# Handoff contains feature cache, authorization, selection artifact, and completed unit JSONs; never the multi-GB WTA cache.
def create_handoff(completed):
    root=Path(RUN_ROOT)
    members=[selection_path,cache_dir/'metadata.json',cache_dir/'train.pt',cache_dir/'test.pt',Path(AUTHORIZATION)]
    output=Path(OUTPUT_DIR)
    members += sorted((output/'units').glob('*.json'))
    members += [path for path in (output/'m18_progress.json',output/'m18_partial_status.json') if path.is_file()]
    hashes={str(path.relative_to(root)).replace('\\','/'):sha_raw(path) for path in members}
    manifest={'schema_version':1,'study_id':'srq-generalization-m18-fly20k-adaptive-locked-test-v1','repo_commit':REPO_COMMIT,'selection_sha256':SELECTION_SHA,'completed_replicates':completed,'files':hashes}
    destination=Path('/content')/f'm18_handoff_{completed:03d}_replicates.zip'
    with zipfile.ZipFile(destination,'w',compression=zipfile.ZIP_STORED,allowZip64=True) as archive:
        for path in members: archive.write(path,str(path.relative_to(root)).replace('\\','/'))
        archive.writestr('HANDOFF_MANIFEST.json',json.dumps(manifest,indent=2)+'\n')
    return destination
print('M18 HANDOFF HELPER READY')

In [ ]:
# Run one new paired replicate per invocation, checkpoint immediately, and free its ~2.2 GB WTA cache.
unit_dir=Path(OUTPUT_DIR)/'units'; unit_dir.mkdir(parents=True,exist_ok=True)
while len(list(unit_dir.glob('replicate_*_paired.json'))) < 6:
    before=len(list(unit_dir.glob('replicate_*_paired.json')))
    print(f'M18 START/RESUME: {before}/6 paired replicates complete',flush=True)
    run_visible([sys.executable,'-B',RUNNER,'run','--config',CONFIG,'--selection-artifact',str(selection_path),'--feature-cache-dir',FEATURE_CACHE_DIR,'--authorization',AUTHORIZATION,'--code-cache-root',CODE_CACHE_ROOT,'--output-dir',OUTPUT_DIR,'--device','cuda','--max-new-replicates','1'])
    after=len(list(unit_dir.glob('replicate_*_paired.json')))
    assert after==before+1,f'Expected one new atomic replicate; got {before} -> {after}'
    completed_unit=json.loads((unit_dir/f'replicate_{after-1}_paired.json').read_text())
    assert completed_unit['status']=='complete',completed_unit.get('failure')
    finished_cache=Path(CODE_CACHE_ROOT)/f'replicate_{after-1}'
    if finished_cache.exists(): shutil.rmtree(finished_cache)
    print(f'M18 CHECKPOINT: {after}/6 complete; finished WTA cache removed')
    if AUTO_DOWNLOAD_HANDOFFS and after in (2,4):
        handoff=create_handoff(after)
        print('DOWNLOAD HANDOFF NOW:',handoff,'SHA256:',sha_raw(handoff),'SIZE:',handoff.stat().st_size)
        files.download(str(handoff))
assert Path(OUTPUT_DIR,'m18_results.json').is_file(),'All units exist but final result was not written.'
print('M18 ALL SIX REPLICATES COMPLETE')

In [ ]:
# Inspect the complete result and export a compact evidence artifact.
result=json.loads(Path(OUTPUT_DIR,'m18_results.json').read_text())
print('STATUS:',result['status'])
print('SUMMARY:',json.dumps(result['summary'],indent=2))
print('GATES:',json.dumps(result['gates'],indent=2))
assert result['status']=='PASS_M18_FLY20K_ADAPTIVE_LOCKED_TEST','Preserve the artifact and report the failed structural gate; do not filter a seed or relax a gate.'
manifest={'schema_version':1,'study_id':result['study_id'],'repo_commit':REPO_COMMIT,'config_sha256':EXPECTED_SOURCE[CONFIG],'runner_sha256':EXPECTED_SOURCE[RUNNER],'selection_sha256':SELECTION_SHA,'result_sha256':sha_raw(Path(OUTPUT_DIR,'m18_results.json'))}
with zipfile.ZipFile(FINAL_EXPORT,'w',compression=zipfile.ZIP_DEFLATED,allowZip64=True) as archive:
    for path in sorted(Path(OUTPUT_DIR).rglob('*')):
        if path.is_file(): archive.write(path,'results/'+str(path.relative_to(OUTPUT_DIR)).replace('\\','/'))
    archive.write(CONFIG,'source/'+CONFIG)
    archive.write('docs/research/SRQ_GENERALIZATION_M18_PROTOCOL.md','source/docs/research/SRQ_GENERALIZATION_M18_PROTOCOL.md')
    archive.writestr('M18_ARTIFACT_MANIFEST.json',json.dumps(manifest,indent=2)+'\n')
print('FINAL EXPORT:',FINAL_EXPORT,'SHA256:',sha_raw(FINAL_EXPORT),'SIZE:',Path(FINAL_EXPORT).stat().st_size)
files.download(FINAL_EXPORT)